In [13]:
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt
import os
from funcs import *
from scipy.special import airy, airye
from cupyx.scipy.sparse import csr_matrix

dir = 'H_1_p'
r, hls, ddrls = load_matrices(dir)
emin = hls[-1,0,0]
n = len(hls[0])
shift = np.array([emin*np.eye(len(hls[0]))]*len(r))
vd = (hls-shift)/4.637
hfunc, ddrfunc = matrix_funcs(r, vd, ddrls)
emax = np.abs(np.array(vd[0,-1,-1]))
h = np.diff(r)/2
w = lambda r, l, e: e*np.eye(n)-(hfunc(r)+l*(l*1)/(r**2)*np.eye(n))
wp = lambda r, l: -(hfunc(r,1)-2*l*(l*1)/(r**3)*np.eye(n))

In [11]:
e_test = emax/4
lmax = int(np.round(np.sqrt(e_test)*np.max(r[-1]))*0.8)
k2 = cp.zeros((len(r)-1, n*lmax))
t = [csr_matrix(cp.zeros((n*lmax, n*lmax), dtype=complex)) for i in range(len(r)-1)]
# p = cp.zeros((len(r)-1, n*lmax, n*lmax), dtype=complex)
wpt = cp.zeros((len(r)-1, n*lmax), dtype=complex)
for i in range(len(r)-1):
    for l in range(lmax):
        k2[i, n*l:n*(l+1)], t[i][n*l:n*(l+1), n*l:n*(l+1)] = cp.linalg.eigh(cp.asarray(w(r[i]+h[i]/2, l, e_test)))
        wpt[i, n*l:n*(l+1)] = cp.diag(t[i][n*l:n*(l+1), n*l:n*(l+1)].H @ cp.asarray(wp(r[i]+h[i]/2, l)) @ t[i][n*l:n*(l+1), n*l:n*(l+1)])
    # if i == 0:
    #     p[i] = t[i]
    # else:
    #     p[i] = cp.transpose(cp.conjugate(t[i])) @ t[i-1]
d1 = cp.zeros((len(r)-1, n*lmax), dtype=complex)
d2 = cp.zeros((len(r)-1, n*lmax), dtype=complex)
d3 = cp.zeros((len(r)-1, n*lmax), dtype=complex)
d4 = cp.zeros((len(r)-1, n*lmax), dtype=complex)
for i in range(len(r)-1):
    alpha = cp.array(cp.cbrt(cp.real(-wpt[i])), dtype=complex)
    beta = k2[i]/wpt[i]
    x = cp.zeros((2, n*lmax), dtype=complex)
    x[0] = alpha*(beta-h[i])
    x[1] = alpha*(beta+h[i])
    ai = cp.zeros((2, n*lmax), dtype=complex)
    aip = cp.zeros((2, n*lmax), dtype=complex)
    bi = cp.zeros((2, n*lmax), dtype=complex)
    bip = cp.zeros((2, n*lmax), dtype=complex)
    dksi = cp.zeros((n*lmax), dtype=complex)
    dksi[cp.real(cp.max(x,axis=0)) > 100] = cp.diff((2./3)*(x[:, cp.real(cp.max(x,axis=0) > 100)]**(3./2)), axis = 0)
    ai[:, cp.real(cp.max(x,axis=0)) > 100], aip[:, cp.real(cp.max(x,axis=0)) > 100], bi[:, cp.real(cp.max(x,axis=0)) > 100], bip[:, cp.real(cp.max(x,axis=0)) > 100] = airye(cp.asnumpy(cp.real(x[:, cp.real(cp.max(x,axis=0)) > 100])))
    ai[:, cp.real(cp.max(x,axis=0)) <= 100], aip[:, cp.real(cp.max(x,axis=0)) <= 100], bi[:, cp.real(cp.max(x,axis=0)) <= 100], bip[:, cp.real(cp.max(x,axis=0)) <= 100] = airy(cp.asnumpy(cp.real(x[:, cp.real(cp.max(x,axis=0)) <= 100])))
    c1 = cp.pi*(cp.exp(-dksi)*cp.asarray(ai[1])*cp.asarray(bip[0])-cp.exp(dksi)*cp.asarray(bi[1])*cp.asarray(aip[0]))
    c2 = cp.pi*(cp.exp(dksi)*cp.asarray(bi[1])*cp.asarray(ai[0])-cp.exp(-dksi)*cp.asarray(ai[1])*cp.asarray(bi[0]))/alpha
    c3 = cp.pi*alpha*(cp.exp(-dksi)*cp.asarray(aip[1])*cp.asarray(bip[0])-cp.exp(dksi)*cp.asarray(bip[1])*cp.asarray(aip[0]))
    c4 = cp.pi*(cp.exp(dksi)*cp.asarray(bip[1])*cp.asarray(ai[0])-cp.exp(-dksi)*cp.asarray(aip[1])*cp.asarray(bi[0]))
    d1[i] = c1 / c2
    d2[i] = 1. / c2
    d3[i] = (c4 * c1 / c2) - c3
    d4[i] = c4 / c2
    print(f'Segment {i} done')
y = [csr_matrix(cp.zeros((n*lmax, n*lmax), dtype=complex)) for i in range(len(r))]
y[0] = 1e308*cp.eye(n*lmax, dtype=complex)
for i in range(len(r)-1):
    prey = cp.diag(d4[i]) - cp.diag(d3[i]) @ cp.linalg.inv(y[i] + cp.diag(d1[i])) @ cp.diag(d2[i])
    if i == 0:
        y[i+1] = t[i] @ prey @ t[i].H
    else:
        y[i+1] = t[i].H @ t[i-1] @ prey @ t[i-1].H @ t[i]
    print(f'Segment {i} done')

# plt.matshow(cp.abs(t[-1].H @ y[-1] @ t[-1]).get())
# plt.colorbar()
# plt.show()

/home/andrey/anaconda3/envs/quantchem/lib/python3.14/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


Segment 0 done
Segment 1 done
Segment 2 done
Segment 3 done
Segment 4 done
Segment 5 done
Segment 6 done
Segment 7 done
Segment 8 done
Segment 9 done
Segment 10 done
Segment 11 done
Segment 12 done
Segment 13 done
Segment 14 done
Segment 15 done
Segment 16 done
Segment 17 done
Segment 18 done
Segment 19 done
Segment 20 done
Segment 21 done
Segment 22 done
Segment 23 done
Segment 24 done
Segment 25 done
Segment 26 done
Segment 27 done
Segment 28 done
Segment 29 done
Segment 30 done
Segment 31 done
Segment 32 done
Segment 33 done
Segment 34 done
Segment 35 done
Segment 36 done
Segment 37 done
Segment 38 done
Segment 39 done
Segment 40 done
Segment 41 done
Segment 42 done
Segment 43 done
Segment 44 done
Segment 45 done
Segment 46 done
Segment 47 done
Segment 48 done
Segment 49 done
Segment 50 done
Segment 51 done
Segment 52 done
Segment 53 done
Segment 54 done
Segment 55 done
Segment 56 done
Segment 57 done
Segment 58 done
Segment 59 done
Segment 60 done
Segment 61 done
Segment 62 done
Se

In [15]:
from airy_props import *
yf = np.zeros((lmax, n, n), dtype=complex)
warr = lambda rs, l, e: np.array([e*np.eye(n)-(hfunc(r)+l*(l*1)/(r**2)*np.eye(n)) for r in rs])
wparr = lambda rs, l: np.array([-(hfunc(r,1)-2*l*(l*1)/(r**3)*np.eye(n)) for r in rs])
for l in range(lmax):
    yf[l] = airy_imb_prop(r, warr(r[:-1]+np.diff(r)/2, l, e_test), wparr(r[:-1]+np.diff(r)/2, l), n)